# FinBERT Full-Data Staged Workflow

Colab-first staged workflow for LM2011-scale FinBERT processing. Sentence parquet is the only v1 handoff artifact between stages: sentence preprocessing, tokenizer profiling, and model inference.

In [ ]:
from pathlib import Path

# Colab bootstrap / install
from google.colab import drive

drive.mount("/content/drive")
#REPO_ROOT = Path("/content/drive/MyDrive/Data_LM/code/NLP_Thesis")
# %cd {REPO_ROOT}
# %pip install -e .[benchmark]

print("If this is a fresh runtime, restart the runtime after the first install.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[Errno 2] No such file or directory: '/content/drive/MyDrive/Data_LM/code/NLP_Thesis'
/content
Obtaining file:///content
ERROR: file:///content does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
If this is a fresh runtime, restart the runtime after the first install.


In [3]:
!rm -rf /content/NLP_Thesis
!pip install "git+https://github.com/Erew42/NLP_Thesis.git@main"

  Cloning https://github.com/Erew42/NLP_Thesis.git (to revision main) to /tmp/pip-req-build-evemivjm
  Running command git clone --filter=blob:none --quiet https://github.com/Erew42/NLP_Thesis.git /tmp/pip-req-build-evemivjm
  Resolved https://github.com/Erew42/NLP_Thesis.git to commit 1d5b6a27ded142037159bf2bcaf45c76b7d1af14
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 872.3/872.3 kB 18.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 900.7/900.7 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.9/75.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18

In [ ]:
from pathlib import Path

FULL_DATA_ROOT = Path("/content/drive/MyDrive/Data_LM")
ITEMS_ANALYSIS_DIR = FULL_DATA_ROOT / "results" / "sec_ccm_unified_runner" / "items_analysis"
BACKBONE_PATH = FULL_DATA_ROOT / "results" / "sec_ccm_unified_runner" / "sec_ccm_premerge" / "final_flagged_data.parquet"
SENTENCE_STAGE_OUT_ROOT = FULL_DATA_ROOT / "results" / "benchmarking" / "finbert_sentence_preprocessing"
TOKENIZER_STAGE_OUT_ROOT = FULL_DATA_ROOT / "results" / "benchmarking" / "finbert_tokenizer_profile"
MODEL_STAGE_OUT_ROOT = FULL_DATA_ROOT / "results" / "benchmarking" / "finbert_sentence_parquet_inference"

YEAR_FILTER = None  # Set to (2006, 2007, 2008) for smoke runs.
RUN_SENTENCE_STAGE = False
RUN_TOKENIZER_STAGE = False
RUN_MODEL_STAGE = False
WRITE_SENTENCE_SCORES = False


In [ ]:
import polars as pl

from thesis_pkg.benchmarking import BucketBatchConfig
from thesis_pkg.benchmarking import FinbertRuntimeConfig
from thesis_pkg.benchmarking import FinbertSectionUniverseConfig
from thesis_pkg.benchmarking import FinbertSentenceParquetInferenceRunConfig
from thesis_pkg.benchmarking import FinbertSentencePreprocessingRunConfig
from thesis_pkg.benchmarking import FinbertTokenizerProfileRunConfig
from thesis_pkg.benchmarking import SentenceDatasetConfig
from thesis_pkg.benchmarking import run_finbert_sentence_parquet_inference
from thesis_pkg.benchmarking import run_finbert_sentence_preprocessing
from thesis_pkg.benchmarking import run_finbert_tokenizer_profile


In [ ]:
SECTION_UNIVERSE = FinbertSectionUniverseConfig(source_items_dir=ITEMS_ANALYSIS_DIR)
SENTENCE_DATASET_CFG = SentenceDatasetConfig(enabled=True, spacy_batch_size=32, token_length_batch_size=1024)
TOKENIZER_BATCH_CFG = BucketBatchConfig(name="baseline", short_batch_size=64, medium_batch_size=32, long_batch_size=16)
MODEL_BATCH_CFG = BucketBatchConfig(name="baseline", short_batch_size=64, medium_batch_size=32, long_batch_size=16)
RUNTIME_CFG = FinbertRuntimeConfig(device=None, use_autocast=True, amp_dtype="auto")

SENTENCE_STAGE_RUN_NAME = "finbert_full_data_sentence_stage"
TOKENIZER_STAGE_RUN_NAME = "finbert_full_data_tokenizer_profile"
MODEL_STAGE_RUN_NAME = "finbert_full_data_model_inference"
SENTENCE_DATASET_DIR = SENTENCE_STAGE_OUT_ROOT / SENTENCE_STAGE_RUN_NAME / "sentence_dataset" / "by_year"


In [ ]:
sentence_stage_artifacts = None
if RUN_SENTENCE_STAGE:
    sentence_stage_artifacts = run_finbert_sentence_preprocessing(
        FinbertSentencePreprocessingRunConfig(
            source_items_dir=ITEMS_ANALYSIS_DIR,
            out_root=SENTENCE_STAGE_OUT_ROOT,
            section_universe=SECTION_UNIVERSE,
            sentence_dataset=SENTENCE_DATASET_CFG,
            target_doc_universe_path=BACKBONE_PATH,
            year_filter=YEAR_FILTER,
            run_name=SENTENCE_STAGE_RUN_NAME,
        )
    )
    print(f"sentence_dataset_dir={sentence_stage_artifacts.sentence_dataset_dir}")
else:
    print("Sentence stage disabled.")


In [ ]:
sentence_summary_path = SENTENCE_STAGE_OUT_ROOT / SENTENCE_STAGE_RUN_NAME / "sentence_dataset_yearly_summary.parquet"
if sentence_summary_path.exists():
    pl.read_parquet(sentence_summary_path)
else:
    print("Sentence stage artifacts not found yet.")


In [ ]:
tokenizer_stage_artifacts = None
if RUN_TOKENIZER_STAGE:
    tokenizer_stage_artifacts = run_finbert_tokenizer_profile(
        FinbertTokenizerProfileRunConfig(
            sentence_dataset_dir=SENTENCE_DATASET_DIR,
            out_root=TOKENIZER_STAGE_OUT_ROOT,
            batch_config=TOKENIZER_BATCH_CFG,
            year_filter=YEAR_FILTER,
            profile_row_cap_per_bucket=5000,
            run_name=TOKENIZER_STAGE_RUN_NAME,
        )
    )
    print(f"timing_summary={tokenizer_stage_artifacts.timing_summary_path}")
else:
    print("Tokenizer stage disabled.")


In [ ]:
tokenizer_summary_path = TOKENIZER_STAGE_OUT_ROOT / TOKENIZER_STAGE_RUN_NAME / "tokenizer_profile_bucket_summary.parquet"
if tokenizer_summary_path.exists():
    pl.read_parquet(tokenizer_summary_path)
else:
    print("Tokenizer profile artifacts not found yet.")


In [ ]:
model_stage_artifacts = None
if RUN_MODEL_STAGE:
    model_stage_artifacts = run_finbert_sentence_parquet_inference(
        FinbertSentenceParquetInferenceRunConfig(
            sentence_dataset_dir=SENTENCE_DATASET_DIR,
            out_root=MODEL_STAGE_OUT_ROOT,
            batch_config=MODEL_BATCH_CFG,
            runtime=RUNTIME_CFG,
            backbone_path=BACKBONE_PATH,
            year_filter=YEAR_FILTER,
            sentence_slice_rows=5000,
            write_sentence_scores=WRITE_SENTENCE_SCORES,
            run_name=MODEL_STAGE_RUN_NAME,
        )
    )
    print(f"item_features_long={model_stage_artifacts.item_features_long_path}")
else:
    print("Model stage disabled.")


In [ ]:
model_summary_path = MODEL_STAGE_OUT_ROOT / MODEL_STAGE_RUN_NAME / "model_inference_yearly_summary.parquet"
item_features_long_path = MODEL_STAGE_OUT_ROOT / MODEL_STAGE_RUN_NAME / "item_features_long.parquet"
if model_summary_path.exists():
    pl.read_parquet(model_summary_path)
elif item_features_long_path.exists():
    pl.read_parquet(item_features_long_path)
else:
    print("Model stage artifacts not found yet.")
